<a href="https://colab.research.google.com/github/likitreddy/book-recommendation-system/blob/main/Book-Recommendation-System3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import pandas as pd
import numpy as np

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
GITHUB_RAW_URL = "https://raw.githubusercontent.com/likitreddy/book-recommendation-system/refs/heads/main/books.csv"
if not os.path.exists("books.csv"):
  !wget -q "{GITHUB_RAW_URL}" -O books.csv
else:
  print("File already exists")
books = pd.read_csv('books.csv')

In [ ]:
books

In [ ]:
ax = plt.axes()
sns.heatmap(books.isna().transpose(), cbar=False, ax=ax)

plt.xlabel("Columes")
plt.ylabel("Missing values")

plt.show()



In [ ]:
books["missing_description"] = np.where(books["description"].isna(), 1, 0)
books["age_of_book"] = 2026 - books["published_year"]

In [ ]:
columns_of_interest = ["num_pages", "age_of_book", "missing_description", "average_rating"]

correlation_matrix = books[columns_of_interest].corr(method ="spearman")

sns.set_theme(style="white")
plt.figure(figsize=(8, 6))
heatmap = sns.heatmap(correlation_matrix, annot=True, fmt=".2f", cmap="coolwarm",
                      cbar_kws={"label": "Spearman correlation"})
heatmap.set_title("Correlation Heatmap")
plt.show()

In [ ]:
book_missing = books[~(books["description"].isna()) &
    ~(books["num_pages"].isna()) &
    ~(books["average_rating"].isna()) &
    ~(books["published_year"].isna())
]

In [ ]:
book_missing["categories"].value_counts().reset_index().sort_values("count", ascending=False)

In [ ]:
book_missing["words_in_description"] = book_missing["description"].str.split().str.len()

In [ ]:
book_missing_25_words = book_missing[book_missing["words_in_description"] >= 25]

In [ ]:
book_missing_25_words["title_and_subtitle"] = (
    np.where(book_missing_25_words["subtitle"].isna(), book_missing_25_words["title"],
             book_missing_25_words[["title", "subtitle"]].astype(str).agg(": ".join, axis=1))
)

In [ ]:
book_missing_25_words["tagged_description"] = book_missing_25_words[["isbn13", "description"]].astype(str).agg(" ".join, axis=1)

In [ ]:
book_missing_25_words

In [ ]:
(
    book_missing_25_words
    .drop(["subtitle", "missing_description", "age_of_book", "words_in_description"], axis=1)
    .to_csv("books_cleaned.csv", index=False)
)

In [ ]:
!pip install -q langchain-huggingface sentence-transformers langchain-chroma langchain-community
from google.colab import userdata
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

In [ ]:
try:
  os.environ["HUGGINGFACEHUB_API_TOKEN"] = userdata.get('HF_TOKEN')
except:
  print("No HF token found")

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

In [ ]:
books = pd.read_csv('books_cleaned.csv')

In [ ]:
books["tagged_description"].to_csv("tagged_description.txt",
                                   sep = "\n",
                                   index = False,
                                   header = False)

In [ ]:
raw_document = TextLoader("tagged_description.txt").load()
text_splitter = CharacterTextSplitter(chunk_size=1, chunk_overlap=0, separator="\n")
documents = text_splitter.split_documents(raw_document)

In [ ]:
db_books = Chroma.from_documents(documents, embeddings)

In [ ]:
docs = db_books.similarity_search("A book to teach children about nature", k=10)
books[books["isbn13"] == int(docs[0].page_content.split()[0].strip())]

In [ ]:
def retrieve_semantic_recommendations(
    query: str,
    top_k: int = 10
) -> pd.DataFrame:
     recs = db_books.similarity_search(query, k=50)

     books_list = []

     for i in range(0, len(recs)):
      books_list += [int(recs[i].page_content.split(' ')[0].strip('"'))]

     return books[books["isbn13"].isin(books_list)].head(top_k)

In [ ]:
retrieve_semantic_recommendations("A book to teach children about nature")

In [ ]:
category_mapping = {'Fiction' : "Fiction",
                    'Juvenile Fiction' : "Children's Fiction",
                    'Biography & Autobiography' : "Nonfiction",
                    'History' :"Nonfiction",
                    'Literary Criticism':"Nonfiction",
                    'Philosophy':"Nonfiction",
                    'Religion':"Nonfiction",
                    'Comics & Graphic Novels':"Fiction",
                    'Drama':"Fiction",
                    'Juvenile Nonfiction': "Children's Nonfiction",
                    'Science':"Nonfiction",
                    'Poetry':"Fiction"}

books["simple_categories"] = books["categories"].map(category_mapping)

In [ ]:
from transformers import pipeline

fiction_categories = ["Fiction", "Nonfiction"]

pipe = pipeline("zero-shot-classification",
                model="facebook/bart-large-mnli",
                device_map="auto")

In [ ]:
sequence = books.loc[books["simple_categories"] == "Fiction", "description"].reset_index(drop=True)[0]

In [ ]:
max_index = np.argmax(pipe(sequence, fiction_categories)["scores"])
max_label = pipe(sequence, fiction_categories)["labels"][max_index]
max_label

In [ ]:
def generate_prediction(sequence, categories):
  prediction = pipe(sequence, categories)
  max_index = np.argmax(prediction["scores"])
  max_label = prediction["labels"][max_index]
  return max_label

In [ ]:
from tqdm import tqdm

actual_cats = []
predicted_cats = []

for i in tqdm(range(0, 300)):
  sequence = books.loc[books["simple_categories"] == "Fiction", "description"].reset_index(drop=True)[i]
  predicted_cats += [generate_prediction(sequence, fiction_categories)]
  actual_cats += ["Fiction"]

In [ ]:
for i in tqdm(range(0, 300)):
  sequence = books.loc[books["simple_categories"] == "Nonfiction", "description"].reset_index(drop=True)[i]
  predicted_cats += [generate_prediction(sequence, fiction_categories)]
  actual_cats += ["Nonfiction"]

In [ ]:
predictions_df = pd.DataFrame({"actual_categories": actual_cats, "predicted_categories": predicted_cats})

In [ ]:
predictions_df["correct_prediction"] = (
    np.where(predictions_df["actual_categories"] == predictions_df["predicted_categories"], 1, 0)
)

In [ ]:
predictions_df["correct_prediction"].sum() / len(predictions_df)

In [ ]:
isbns = []
predicted_cats = []

missing_cats =  books.loc[books["simple_categories"].isna(), ["isbn13", "description"]].reset_index(drop=True)

In [ ]:
for i in tqdm(range(0, len(missing_cats))):
  sequence = missing_cats["description"][i]
  predicted_cats += [generate_prediction(sequence, fiction_categories)]
  isbns += [missing_cats["isbn13"][i]]

In [ ]:
missing_predicted_df = pd.DataFrame({"isbn13": isbns, "predicted_categories": predicted_cats})

In [ ]:
books = pd.merge(books, missing_predicted_df, on="isbn13", how="left")
books["simple_categories"] = np.where(books["simple_categories"].isna(), books["predicted_categories"], books["simple_categories"])
books = books.drop(columns=["predicted_categories"])

In [ ]:
books[books["categories"].str.lower().isin([
    "romance",
    "science fiction",
    "scifi",
    "fantasy",
    "horror",
    "mystery",
    "thiller",
    "comedy",
    "crime",
    "historical"
])]

In [ ]:
books.to_csv("books_with_categories.csv", index=False)

In [ ]:
books = pd.read_csv("books_with_categories.csv")

In [ ]:
from transformers import pipeline
classifier = pipeline("text-classification",
                      model="j-hartmann/emotion-english-distilroberta-base",
                      return_all_scores=True)
classifier("I love this!")

In [ ]:
books["description"][0]

In [ ]:
classifier(books["description"][0])

In [ ]:
classifier(books["description"][0].split("."))

In [ ]:
sentences = books["description"][0].split(".")
predictions = classifier(sentences)

In [ ]:
predictions

In [ ]:
sorted(predictions, key=lambda x: x["label"])

In [ ]:
emotion_labels = ["anger", "disgust", "fear", "joy", "sadness", "surprise", "neutral"]
isbn = []
emotion_score = {label: [] for label in emotion_labels}

def calculate_max_emption_score(predictions):
    max_scores_for_description = {label: -np.inf for label in emotion_labels}

    for pred_item in predictions:
        label = pred_item['label']
        score = pred_item['score']
        if label in max_scores_for_description:
            if score > max_scores_for_description[label]:
                max_scores_for_description[label] = score

    for label in max_scores_for_description:
        if max_scores_for_description[label] == -np.inf:
            max_scores_for_description[label] = 0.0

    return max_scores_for_description

In [ ]:
for i in range(10):
  isbn.append(books["isbn13"][i])
  sentences = books["description"][i].split(".")
  sentences = [s.strip() for s in sentences if s.strip()]

  if sentences:
    predictions = classifier(sentences)
    max_scores = calculate_max_emption_score(predictions)
    for label in emotion_labels:
      emotion_score[label].append(max_scores[label])
  else:
    for label in emotion_labels:
        emotion_score[label].append(0.0)

In [ ]:
emotion_score

In [ ]:
from tqdm import tqdm

emotion_labels = ["anger", "disgust", "fear", "joy", "sadness", "surprise", "neutral"]
isbn = []
emotion_score = {label: [] for label in emotion_labels}

for i in tqdm(range(0, len(books))):
  isbn.append(books["isbn13"][i])
  sentences = books["description"][i].split(".")
  predictions = classifier(sentences)
  max_scores = calculate_max_emption_score(predictions)
  for label in emotion_labels:
    emotion_score[label].append(max_scores[label])

In [ ]:
emotions_df = pd.DataFrame(emotion_score)
emotions_df["isbn13"] = isbn

In [ ]:
books = pd.merge(books, emotions_df, on="isbn13")

In [ ]:
books

In [ ]:
books.to_csv("books_with_emotions.csv", index = False)

In [ ]:
import pandas as pd
import numpy as np

from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_chroma import Chroma

!pip install --upgrade gradio
import gradio as gr

from langchain_huggingface import HuggingFaceEmbeddings

books = pd.read_csv("books_with_emotions.csv")
books["large_thumbnail"] = books["thumbnail"] + "&fife=w800"
books["large_thumbnail"] = np.where(
    books["thumbnail"].isna(),
    "cover-not-found.jpg",
    books["large_thumbnail"],
    )

documents = TextLoader("tagged_description.txt").load()

db_books = Chroma.from_documents(documents, HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2"))


def retrieve_semantic_recommendations(
    query: str,
    category: str = None,
    initial_top_k: int = 50,
    final_top_k: int = 16,
    tone: str = None,
) -> pd.DataFrame:

     recs = db_books.similarity_search(query, k=initial_top_k)
     books_list = [int(rec.page_content.split(' ')[0].strip('"')) for rec in recs]
     book_recs = books[books["isbn13"].isin(books_list)].head(final_top_k)

     if category != "All":
      book_recs = book_recs[book_recs["simple_categories"] == category].head(final_top_k)
     else:
      book_recs = book_recs.head(final_top_k)

     if tone == "Happy":
      book_recs = book_recs.sort_values(by="joy", ascending=False)
     elif tone == "Surprising":
      book_recs = book_recs.sort_values(by="surprise", ascending=False)
     elif tone == "Angry":
      book_recs = book_recs.sort_values(by="anger", ascending=False)
     elif tone == "Suspenseful":
      book_recs = book_recs.sort_values(by="fear", ascending=False)
     elif tone == "Sad":
      book_recs = book_recs.sort_values(by="sadness", ascending=False)

     return book_recs


def recommend_books(
    query,
    category,
    tone
):

    recommendations = retrieve_semantic_recommendations(
        query=query,
        category=category,
        tone=tone,
        initial_top_k=50,
        final_top_k=16)
    result = []

    for _, row in recommendations.iterrows():
        description = row["description"]
        truncated_desc_split = description.split()
        truncated_description = " ".join(truncated_desc_split[:30]) + "..."

        authors_split = row["authors"].split(";")
        if len(authors_split) == 2:
            authors_str = f"{authors_split[0]} and {authors_split[1]}"
        elif len(authors_split) > 2:
            authors_str = f"{', '.join(authors_split[:-1])} and {authors_split[-1]}"
        else:
            authors_str = row["authors"]

        caption = f"{row['title']} by {authors_str}: {truncated_description}"
        result.append(row["large_thumbnail"])
        result.append(caption)

    return result

categories = ["All"] + sorted(books["simple_categories"].unique())
tones = ["All"] + ["Happy", "Surprising", "Angry", "Suspenseful", "Sad"]

with gr.Blocks(theme = gr.themes.Glass()) as dashboard:
  gr.Markdown("# Semantic book recommender")

  with gr.Row():
    user_query = gr.Textbox(label="Please enter a description of a book",
                            placeholder = "e.g., A story about forgiveness")
    category_dropdown = gr.Dropdown(label="Select a category",
                                    choices=categories,
                                    value="All")
    tone_dropdown = gr.Dropdown(label="Select a emotional tone",
                                 choices=tones,
                                 value="All")
    submit_button = gr.Button("Find recommendations")

  gr.Markdown("## Recommendations")
  output = gr.Gallery(label="Recommended books",
                      columns = 8,
                      rows = 2)
  submit_button.click(fn = recommend_books,
                      inputs = [user_query, category_dropdown, tone_dropdown],
                      outputs = output)

if __name__ == "__main__":
  dashboard.launch()